# 10 — Approach D: synthetic data from physics → pretrain → fine-tune

Code: `src/synthetic.py`. Experiments are run (in parallel, ~25 min) by:
```
python src/run_synthetic_experiments.py
```
which saves `outputs/synthetic_results.csv` and `outputs/synthetic_predictions.parquet`. This notebook analyses them.

**Simulator** (re-calibrated inside every validation fold on the training tests only → no leakage):
- inputs: the 2,725 real flowing daily sensor rows, jittered by ~1 % (→ nearby operating points, e.g. Pwf ± 30 psi)
- labels: physics `q_oil = J·(P_res(t) − Pwf)`, water from WC trend, gas from GOR, **+ measurement noise**
  of the size we measured (oil 2.5 %, water 4 %, gas 6 %)

**Training modes**

| Mode | What | Models |
|---|---|---|
| real_only | 84 real tests only (from notebook 08; MLP new here) | all |
| synthetic_only | synthetic only — can ML learn the physics from examples? | all |
| finetune | pretrain on synthetic, then **continue training** on real (boosting: +100 trees on real; neural net: +100 epochs on real) | LightGBM, XGBoost, CatBoost, MLP |
| mixed | synthetic + real together, real rows repeated so both have equal total weight | all |

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path('..', 'src').resolve()))
import load
import validation as V
import vfm

OUT = load.ROOT / 'outputs'
tt = pd.read_parquet(load.PROCESSED / 'test_features.parquet')
syn_results = pd.read_csv(OUT / 'synthetic_results.csv')
syn_preds = pd.read_parquet(OUT / 'synthetic_predictions.parquet')
all_results = V.save_results(syn_results.to_dict('records'), OUT / 'ablation_results.csv')

# real-only scores of the same models from notebook 08 (physics features + one-hot well id)
real = all_results[all_results.experiment.str.match(r'A .* physics feats$')].copy()
real['model'] = real.experiment.str.split().str[1]
real['mode'] = 'real_only'
syn = syn_results.copy()
syn['model'] = syn.experiment.str.split().str[1]
syn['mode'] = syn.experiment.str.split().str[2]

table = pd.concat([real, syn]).pivot_table(index='model', columns='mode', values='mean_MAPE')
table = table.loc[table.index.isin(syn.model)]      # only models used in the synthetic experiments
table = table[['real_only', 'synthetic_only', 'finetune', 'mixed']].sort_values('mixed')
print('mean MAPE % (oil, water, gas).  Physics = 4.04,  best so far (E2 ensemble) = 3.84')
table.round(2)

In [ ]:
ax = table.plot.barh(figsize=(10, 6))
ax.axvline(4.04, color='green', ls='--', label='physics (4.04)')
ax.axvline(3.84, color='black', ls=':', label='E2 ensemble (3.84)')
ax.set_xlabel('mean MAPE % — lower is better')
ax.set_title('Effect of synthetic pretraining per model')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Per-target results of every synthetic experiment

In [ ]:
syn_results[['experiment', 'oil_MAPE', 'water_MAPE', 'gas_MAPE', 'mean_MAPE']].sort_values('mean_MAPE').round(2)

## Best synthetic experiments vs physics and vs the E2 ensemble (paired tests)

In [ ]:
def get_predictions(name):
    return syn_preds[syn_preds.experiment == name].set_index('test_index')[V.TARGETS].sort_index()

physics_pred = V.leave_one_test_out(tt, vfm.physics)
gp = vfm.make_ml_method(vfm.PHYSICS_FEATURES, vfm.gaussian_process, use_well_id=True, well_encoding='onehot')
weights = {'oil_rate_stbd': 0, 'water_rate_stbd': 0.5, 'gas_rate_mscfd': 0}
e2_pred = V.leave_one_test_out(tt, vfm.make_blend_method(vfm.physics, gp, weights))

for name in syn_results.sort_values('mean_MAPE').experiment.head(3):
    print(f'=== {name}')
    print('vs physics:');     print(V.paired_test(tt, physics_pred, get_predictions(name)).round(3))
    print('vs E2 ensemble:'); print(V.paired_test(tt, e2_pred, get_predictions(name)).round(3), '\n')

**Conclusions**
- **Synthetic data rescues the tree models.** Boosting / forests were the worst on 84 real rows; with ~2,700 synthetic
  rows they improve a lot: LightGBM 8.13 -> 5.05, XGBoost 8.77 -> 4.99, CatBoost 6.24 -> 4.63,
  RandomForest 7.77 -> 5.04, ExtraTrees 5.95 -> 4.60. MLP (neural net) 6.65 -> 5.66 with fine-tuning.
- **Fine-tune vs mixed:** similar overall; fine-tuning is best for CatBoost and XGBoost.
- **Smooth models get worse** (GP 4.26 -> 9.05, Ridge 4.94 -> 6.10, kNN 5.48 -> 5.94). Ridge cannot represent the
  per-well physics (J differs per well -> interactions a linear model doesn't have); the GP only gets a 300-row
  subsample and is pulled away from the real tests it previously interpolated well.
- **No synthetic experiment beats physics (4.04) or the E2 ensemble (3.84)**; the best ones are significantly worse
  on oil and gas (p < 0.05). Expected: the synthetic data contains only what the physics already knows, so the
  best an ML model can do is copy the physics — and it copies it imperfectly.
- **Take-away for the presentation:** physics-generated data is a valid way to make data-hungry models usable on
  tiny datasets (trees improve by 25-40 %), but when the physics itself is available, using it directly is better.
- Final VFM model stays **E2: physics oil & gas + 50/50 physics/GP water.**
